# Benchmark Results

Reads `data/baseline/results.csv` and `results_by_similarity.csv`. Nothing is retrained.
Test split throughout. RMSE: lower is better; everything else: higher is better.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

AI_DIR = Path.cwd()
if AI_DIR.name == "notebooks":
    AI_DIR = AI_DIR.parent
elif AI_DIR.name != "ai":
    AI_DIR = AI_DIR / "ai"
sys.path.insert(0, str(AI_DIR / "src"))

from baseline.results import (
    DEFAULT_SPLIT_ID,
    load_results,
    summarize_seeds,
    summarize_splits,
)
from baseline.compare import PAIRED_PATH
from baseline.similarity import SIMILARITY_PATH

HEADLINE = [
    ("pIC50_BACE1", "rmse"),
    ("pIC50_EGFR", "rmse"),
    ("pIC50_BACE1", "spearman"),
    ("pIC50_EGFR", "spearman"),
    ("macro_tox21", "roc_auc"),
    ("macro_tox21", "pr_auc"),
    ("CT_TOX", "roc_auc"),
    ("CT_TOX", "pr_auc"),
    ("FDA_APPROVED", "roc_auc"),
]
COLORS = {"mufu": "tab:blue", "rf": "tab:orange"}
EXTRA_COLORS = iter(["tab:green", "tab:red", "tab:purple", "tab:brown", "tab:pink"])


def family(model):
    return "rf" if model.startswith("rf") else model


def color(model):
    key = family(model)
    if key not in COLORS:
        COLORS[key] = next(EXTRA_COLORS, "tab:gray")
    return COLORS[key]


results = load_results()
test = results[results["split"] == "test"]


def headline_rows(df):
    return pd.MultiIndex.from_frame(df[["endpoint", "metric"]]).isin(HEADLINE)


headline = headline_rows(test)

## Validation: Compare Changes Here

Default split, mean ± std over seeds. Pick between MuFu variants on these numbers. The test
sections below are for the final version only: choosing whichever variant scores best on
test makes its test score optimistic.

In [ ]:
val = results[(results["split"] == "val") & (results["split_id"] == DEFAULT_SPLIT_ID)]
val = summarize_seeds(val[headline_rows(val)])
cell = val["value"].map("{:.3f}".format) + " ± " + val["std"].map("{:.3f}".format)
val.assign(model=val["model"].map(family), cell=cell).pivot(
    index=["endpoint", "metric"], columns="model", values="cell"
)

## Default Split with 95% Bootstrap CIs

`std` is across seeds (training noise only). `ci_low`/`ci_high` resample test compounds, so
they show how much the number would move on a different test set of the same size.

In [ ]:
default = summarize_seeds(test[headline & (test["split_id"] == DEFAULT_SPLIT_ID)])
columns = ["model", "endpoint", "metric", "value", "std", "ci_low", "ci_high", "n_seeds"]
default[columns].round(3)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 9))
for ax, (endpoint, metric) in zip(axes.flat, HEADLINE):
    rows = default[(default["endpoint"] == endpoint) & (default["metric"] == metric)]
    for i, r in enumerate(rows.itertuples()):
        ax.errorbar(
            i,
            r.value,
            yerr=[[r.value - r.ci_low], [r.ci_high - r.value]],
            fmt="o",
            capsize=6,
            color=color(r.model),
        )
    ax.set_xticks(range(len(rows)), [family(m) for m in rows["model"]])
    ax.set_xlim(-0.6, len(rows) - 0.4)
    ax.set_title(f"{endpoint} {metric}")
    ax.grid(axis="y", alpha=0.3)
fig.suptitle("Default split, seed mean with 95% bootstrap CI")
fig.tight_layout()
plt.show()

## Paired Model − RF

From `python -m baseline.compare [model]`. Each of 2000 resamples draws the same test
compounds for the model and RF and averages the delta over seeds, so the CI covers test
sampling and training noise together. `p_a_better` is the share of resamples where the
model beats RF. Default split only. In the plot, right of 0 is better than RF for every
metric.

In [ ]:
paired = pd.read_csv(PAIRED_PATH)
display(paired.drop(columns=["split_id", "n_seeds"]).round(3))

# flip RMSE so right of 0 is better than RF everywhere
flip = paired["metric"] == "rmse"
paired["d"] = paired["delta"].where(~flip, -paired["delta"])
paired["lo"] = paired["ci_low"].where(~flip, -paired["ci_high"])
paired["hi"] = paired["ci_high"].where(~flip, -paired["ci_low"])
paired["label"] = paired["endpoint"] + " " + paired["metric"]
labels = list(dict.fromkeys(paired["label"]))
fig, ax = plt.subplots(figsize=(7, 4))
for k, (model, g) in enumerate(paired.groupby("a", sort=False)):
    y = [len(labels) - 1 - labels.index(label) - 0.2 * k for label in g["label"]]
    xerr = [g["d"] - g["lo"], g["hi"] - g["d"]]
    ax.errorbar(g["d"], y, xerr=xerr, fmt="o", capsize=4, color=color(model), label=model)
ax.axvline(0, color="gray", lw=1)
ax.set_yticks(range(len(labels)), labels[::-1])
ax.set_xlabel("model − RF, 95% paired bootstrap CI")
ax.legend()
fig.tight_layout()
plt.show()

## RF Across Scaffold Splits

`split_std` is the std of the six split means, `seed_std` the mean std across seeds within
a split. `default` puts the largest scaffolds in train, so it's the hardest split.

In [ ]:
rf = test[test["model"].str.startswith("rf") & headline]
summarize_splits(rf)[
    ["endpoint", "metric", "value", "split_std", "seed_std", "n_splits"]
].round(3)

In [ ]:
panels = [
    ("pIC50_BACE1", "rmse"),
    ("pIC50_EGFR", "rmse"),
    ("macro_tox21", "roc_auc"),
    ("FDA_APPROVED", "roc_auc"),
]
fig, axes = plt.subplots(1, len(panels), figsize=(15, 4))
for ax, (endpoint, metric) in zip(axes, panels):
    rows = rf[(rf["endpoint"] == endpoint) & (rf["metric"] == metric)]
    order = sorted(rows["split_id"].unique(), key=lambda s: (s != DEFAULT_SPLIT_ID, s))
    for i, split_id in enumerate(order):
        values = rows.loc[rows["split_id"] == split_id, "value"]
        ax.scatter([i] * len(values), values, s=12, color=color("rf"), alpha=0.6)
    means = [rows.loc[rows["split_id"] == s, "value"].mean() for s in order]
    ax.plot(range(len(order)), means, color="gray", lw=1)
    ax.set_xticks(range(len(order)), [s.replace("scaffold-", "s") for s in order])
    ax.set_title(f"RF {endpoint} {metric}")
    ax.grid(axis="y", alpha=0.3)
fig.suptitle("RF over scaffold splits, one dot per seed")
fig.tight_layout()
plt.show()

## Distance to the Training Set

Test compounds bucketed by nearest-neighbour ECFP4 Tanimoto to train, default split, mean ±
std over seeds. `n` is labeled compounds for that endpoint (all test compounds for macros).
Low-similarity buckets are small, and a bucket with one class has no ROC-AUC.

In [ ]:
sim = pd.read_csv(SIMILARITY_PATH)
panels = [
    ("pIC50_BACE1", "rmse"),
    ("pIC50_EGFR", "rmse"),
    ("macro_tox21", "roc_auc"),
    ("CT_TOX", "roc_auc"),
]
fig, axes = plt.subplots(1, len(panels), figsize=(16, 4))
for ax, (endpoint, metric) in zip(axes, panels):
    rows = sim[(sim["endpoint"] == endpoint) & (sim["metric"] == metric)]
    buckets = sorted(rows["bucket"].unique())
    for model, g in rows.groupby("model"):
        g = g.set_index("bucket").reindex(buckets)
        ax.errorbar(
            range(len(buckets)),
            g["value"],
            yerr=g["std"],
            marker="o",
            capsize=4,
            color=color(model),
            label=family(model),
        )
    n = rows.groupby("bucket")[["n_labeled", "n_compounds"]].first()
    n = n["n_labeled"].fillna(n["n_compounds"]).reindex(buckets)
    labels = [f"{b}\nn={int(c)}" for b, c in zip(buckets, n)]
    ax.set_xticks(range(len(buckets)), labels, fontsize=7)
    ax.set_title(f"{endpoint} {metric}")
    ax.grid(axis="y", alpha=0.3)
    ax.legend()
fig.suptitle("Test metric vs nearest-train Tanimoto")
fig.tight_layout()
plt.show()